# Narrative intensity × APWC × Residual Momentum × RRG 分析ノートブック

このノートブックは、日次の narrative intensity が既に作成済みである前提で、以下を一気通貫で実行します。

1. daily narrative intensity から attention shock / z-score shock を作成  
2. テーマ別の Barra 残差リターン、APWC、Residual Momentum、RRG 指標を整備  
3. テーマ別 narrative beta を rolling 推定  
4. `NarrativeSignal = Σ narrative beta × narrative shock` を作成  
5. Narrative Signal が将来の APWC、残差リターン、Residual Momentum、RRG 遷移に先行するか検証  
6. クロスセクション・ソート、イベントスタディ、簡易ポートフォリオ分析を実行  

理論的な対応関係は以下です。

- **Candes et al. 型 APWC**：テーマ内銘柄の Barra 残差リターンが共通に動いているか  
- **Residual Momentum**：テーマ固有成分に価格トレンドがあるか  
- **Narrative Momentum / MKT Multi-Asset Rotation**：ニュース上の narrative attention shock が先行情報になるか  
- **RRG**：相対力と相対力モメンタムからテーマのライフサイクル局面を可視化する  

実データが指定パスに存在しない場合は、構造を確認できるように synthetic demo data を自動生成します。

## 0. 入力データ仕様

このノートブックは、以下のどちらかの形式でテーマ側データを受け取れます。

### A. 推奨：個別銘柄 Barra 残差リターンから APWC とテーマ残差を計算する場合

`constituent_residual_returns.csv` または `.parquet`

必須列:

```text
date, theme_id, security_id, residual_return
```

任意列:

```text
weight
```

`weight` がない場合は、テーマ内等ウェイトで合成します。  
この形式がある場合、ノートブック内で以下を作ります。

\[
\epsilon_{g,t}=\sum_{i\in g}w_{i,g,t-1}\epsilon_{i,t}
\]

\[
APWC_{g,t}=\frac{2}{N_g(N_g-1)}\sum_{i<j}\rho(\epsilon_i,\epsilon_j)
\]

### B. 既にテーマ別の残差リターンとAPWCがある場合

`theme_residual_returns.csv`

```text
date, theme_id, residual_return
```

`theme_apwc.csv`

```text
date, theme_id, APWC
```

### Narrative intensity

`daily_narrative_intensity.csv`

long format の例:

```text
date, narrative_id, I_negative, I_total, I_positive
```

または、wide format でも読み込めます。

```text
date, AI, Inflation, Recession, Defense, ...
```

その場合、各 narrative 列が intensity として melt されます。

## 1. インポートと基本設定

`CONFIG` セルで、データ保存場所、使う intensity 列、rolling window、予測 horizon などを指定します。

- `INTENSITY_COL`: 主分析に使う intensity。通常は `I_negative` を推奨します。
- `SHOCK_WINDOWS`: daily intensity から作る attention shock の期間。20D, 60D, 120D が基本です。
- `BETA_WINDOW`: テーマ別 narrative beta の rolling 推定期間。
- `APWC_WINDOW`: constituent residual returns から APWC を計算する場合の rolling 窓。
- `FUTURE_HORIZONS`: 将来APWCや将来残差リターンを見る horizon。

In [ ]:
from __future__ import annotations

from pathlib import Path
from dataclasses import dataclass
from typing import Iterable, Optional, Sequence
import warnings
import zipfile
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import statsmodels.api as sm
    HAS_STATSMODELS = True
except Exception:
    HAS_STATSMODELS = False
    warnings.warn("statsmodels が見つかりません。回帰分析は簡易OLSにフォールバックします。")

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

@dataclass
class Config:
    # ===== Paths =====
    DATA_DIR: Path = Path("./data")
    OUTPUT_DIR: Path = Path("./outputs")

    NARRATIVE_INTENSITY_FILE: str = "daily_narrative_intensity.csv"
    CONSTITUENT_RESIDUAL_FILE: str = "constituent_residual_returns.csv"
    THEME_RESIDUAL_FILE: str = "theme_residual_returns.csv"
    THEME_APWC_FILE: str = "theme_apwc.csv"

    # auto: real files があれば実データ、なければdemo data
    USE_DEMO_DATA: str | bool = "auto"

    # ===== Column names =====
    DATE_COL: str = "date"
    NARRATIVE_COL: str = "narrative_id"
    THEME_COL: str = "theme_id"
    SECURITY_COL: str = "security_id"
    RESIDUAL_RETURN_COL: str = "residual_return"
    WEIGHT_COL: str = "weight"
    APWC_COL: str = "APWC"

    # 主分析に使う narrative intensity
    # 候補: I_negative, I_total, I_positive, intensity
    INTENSITY_COL: str = "I_negative"

    # ===== Narrative signal construction =====
    SHOCK_WINDOWS: tuple[int, ...] = (20, 60, 120)     # 営業日換算: 約1M, 3M, 6M
    SHOCK_Z_WINDOW: int = 252
    SHOCK_Z_MIN_PERIODS: int = 126
    BETA_DRIVER_COL: str = "D1"                       # narrative beta 推定用。D1 = daily change
    BETA_WINDOW: int = 252
    BETA_MIN_PERIODS: int = 126
    MAX_NARRATIVES: int = 80                          # 多すぎる場合は平均 intensity 上位に絞る
    MIN_NARRATIVE_OBS: int = 252

    # ===== Theme features =====
    APWC_WINDOW: int = 60
    APWC_MIN_OBS: int = 30
    APWC_Z_WINDOW: int = 252
    RMOM_WINDOWS: tuple[int, ...] = (20, 60, 120)
    VOL_WINDOWS: tuple[int, ...] = (20, 60)
    FUTURE_HORIZONS: tuple[int, ...] = (5, 20, 60)

    # ===== RRG =====
    RRG_WINDOW: int = 120
    RRG_MOM_LAG: int = 20

    # ===== Regressions =====
    MAIN_SIGNAL_COL: str = "narr_signal_60D"
    MAIN_RMOM_COL: str = "rmom_60D"
    MAIN_FUTURE_RETURN_COL: str = "fut_resid_ret_20D"
    MAIN_FUTURE_APWC_DELTA_COL: str = "fut_apwc_z_delta_20D"

CONFIG = Config()
CONFIG.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CONFIG

## 2. 汎用ユーティリティ

ここでは、CSV / ZIP / Parquet の読込、列名の正規化、rolling 計算、将来リターン計算などを定義します。

日次ZIPのようなファイルは、`pd.read_csv(..., compression="zip")` でも読めることが多いですが、このノートブックでは日次 intensity が既に集計済みである前提なので、基本的には `daily_narrative_intensity.csv` を読み込みます。

In [ ]:
def read_table(path: Path) -> pd.DataFrame:
    """Read csv, csv.zip, or parquet."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)

    suffixes = "".join(path.suffixes).lower()
    if suffixes.endswith(".parquet"):
        return pd.read_parquet(path)
    if suffixes.endswith(".csv.zip") or suffixes.endswith(".zip"):
        return pd.read_csv(path, compression="zip")
    if suffixes.endswith(".csv"):
        return pd.read_csv(path)
    raise ValueError(f"Unsupported file type: {path}")

def parse_date_col(df: pd.DataFrame, date_col: str = "date") -> pd.DataFrame:
    out = df.copy()
    out[date_col] = pd.to_datetime(out[date_col])
    return out

def normalize_id_columns(df: pd.DataFrame, config: Config) -> pd.DataFrame:
    """Common aliases を標準列名に寄せる。"""
    out = df.copy()

    aliases = {
        config.DATE_COL: ["date", "datetime", "timestamp", "file_date"],
        config.NARRATIVE_COL: ["narrative_id", "narrative", "topic", "topic_id", "theme_narrative"],
        config.THEME_COL: ["theme_id", "theme", "basket", "basket_id"],
        config.SECURITY_COL: ["security_id", "ticker", "ric", "sedol", "permno"],
        config.RESIDUAL_RETURN_COL: ["residual_return", "resid_return", "barra_residual_return", "epsilon"],
        config.APWC_COL: ["APWC", "apwc", "average_pairwise_corr"],
        config.WEIGHT_COL: ["weight", "w", "basket_weight"],
    }

    lower_map = {c.lower(): c for c in out.columns}
    for target, candidates in aliases.items():
        if target in out.columns:
            continue
        for cand in candidates:
            if cand.lower() in lower_map:
                out = out.rename(columns={lower_map[cand.lower()]: target})
                break
    return out

def rolling_zscore(s: pd.Series, window: int, min_periods: int) -> pd.Series:
    m = s.rolling(window, min_periods=min_periods).mean()
    sd = s.rolling(window, min_periods=min_periods).std()
    return (s - m) / sd.replace(0, np.nan)

def past_sum(s: pd.Series, window: int) -> pd.Series:
    """Sum of past window observations excluding current date."""
    return s.shift(1).rolling(window, min_periods=max(5, window // 3)).sum()

def past_vol(s: pd.Series, window: int) -> pd.Series:
    return s.shift(1).rolling(window, min_periods=max(5, window // 3)).std()

def future_sum(s: pd.Series, horizon: int) -> pd.Series:
    """At t, sum returns from t+1 to t+horizon."""
    out = pd.Series(0.0, index=s.index)
    for k in range(1, horizon + 1):
        out = out + s.shift(-k)
    return out

def safe_log1p(x: pd.Series) -> pd.Series:
    return np.log1p(x.clip(lower=-0.999999))

## 3. Demo data 生成

指定された実データが存在しない場合でも、ノートブック全体が動くように synthetic demo data を作ります。

実データで実行する場合は、`CONFIG.DATA_DIR` に必要ファイルを配置してください。  
デモデータは、narrative shock が一部テーマの残差リターンとAPWCに先行するように作ってあります。

In [ ]:
def compute_demo_shocks(intensity_long: pd.DataFrame, config: Config, intensity_col: str = "I_negative") -> pd.DataFrame:
    df = intensity_long.sort_values([config.NARRATIVE_COL, config.DATE_COL]).copy()
    df["D1"] = df.groupby(config.NARRATIVE_COL)[intensity_col].diff()

    for w in config.SHOCK_WINDOWS:
        recent = df.groupby(config.NARRATIVE_COL)[intensity_col].transform(
            lambda x: x.shift(1).rolling(w, min_periods=max(5, w // 3)).mean()
        )
        prev = df.groupby(config.NARRATIVE_COL)[intensity_col].transform(
            lambda x: x.shift(1 + w).rolling(w, min_periods=max(5, w // 3)).mean()
        )
        df[f"Shock_{w}D"] = recent - prev
        df[f"ZShock_{w}D"] = df.groupby(config.NARRATIVE_COL)[f"Shock_{w}D"].transform(
            lambda x: rolling_zscore(x, config.SHOCK_Z_WINDOW, config.SHOCK_Z_MIN_PERIODS)
        )
    return df

def generate_demo_data(config: Config):
    rng = np.random.default_rng(42)
    dates = pd.bdate_range("2019-01-01", "2024-12-31")
    narratives = [
        "AI", "Semiconductors", "Cloud", "Inflation", "InterestRates",
        "Defense", "EnergySecurity", "Nuclear", "Recession", "Liquidity"
    ]
    themes = [
        "AI_Basket", "Semi_Basket", "Cloud_Software", "Banks", "Defense_Basket",
        "Nuclear_Power", "Utilities", "Energy", "Travel_Reopening", "Small_Growth"
    ]

    # narrative intensity: positive AR-like process with several narrative waves
    rows = []
    for n in narratives:
        level = 0.01 + 0.02 * rng.random()
        x = []
        val = level
        for i, d in enumerate(dates):
            shock = rng.normal(0, 0.001)
            val = 0.96 * val + 0.04 * level + shock
            val = max(val, 0.0001)

            # artificial waves
            if n in ["AI", "Semiconductors", "Cloud"] and pd.Timestamp("2023-01-01") <= d <= pd.Timestamp("2024-06-30"):
                val += 0.00015
            if n in ["Defense", "EnergySecurity"] and pd.Timestamp("2022-02-01") <= d <= pd.Timestamp("2023-06-30"):
                val += 0.00012
            if n in ["Inflation", "InterestRates"] and pd.Timestamp("2021-06-01") <= d <= pd.Timestamp("2023-03-31"):
                val += 0.00010
            x.append(val)
        x = np.array(x)
        x = np.clip(x, 0, None)
        for d, val in zip(dates, x):
            rows.append({
                config.DATE_COL: d,
                config.NARRATIVE_COL: n,
                "I_negative": val,
                "I_total": val * (2.0 + rng.normal(0, 0.1)),
                "I_positive": val * (0.7 + rng.normal(0, 0.05)),
            })
    intensity = pd.DataFrame(rows)
    intensity = compute_demo_shocks(intensity, config, "I_negative")

    # true theme-narrative links
    true_gamma = pd.DataFrame(0.0, index=themes, columns=narratives)
    true_gamma.loc["AI_Basket", ["AI", "Semiconductors", "Cloud"]] = [0.0018, 0.0010, 0.0008]
    true_gamma.loc["Semi_Basket", ["Semiconductors", "AI"]] = [0.0020, 0.0007]
    true_gamma.loc["Cloud_Software", ["Cloud", "AI"]] = [0.0018, 0.0007]
    true_gamma.loc["Banks", ["InterestRates", "Liquidity"]] = [0.0015, -0.0010]
    true_gamma.loc["Defense_Basket", ["Defense", "EnergySecurity"]] = [0.0018, 0.0006]
    true_gamma.loc["Nuclear_Power", ["Nuclear", "EnergySecurity"]] = [0.0017, 0.0008]
    true_gamma.loc["Utilities", ["InterestRates", "Nuclear", "EnergySecurity"]] = [-0.0008, 0.0005, 0.0005]
    true_gamma.loc["Energy", ["EnergySecurity", "Inflation"]] = [0.0014, 0.0006]
    true_gamma.loc["Travel_Reopening", ["Recession", "EnergySecurity"]] = [-0.0010, -0.0005]
    true_gamma.loc["Small_Growth", ["Liquidity", "InterestRates", "AI"]] = [0.0012, -0.0012, 0.0004]

    zshock_wide = intensity.pivot(index=config.DATE_COL, columns=config.NARRATIVE_COL, values="ZShock_60D").fillna(0.0)

    # theme residual returns respond to lagged narrative shocks + noise
    theme_rows = []
    apwc_rows = []
    for theme in themes:
        sig = zshock_wide.reindex(columns=narratives).fillna(0.0).values @ true_gamma.loc[theme].values
        sig = pd.Series(sig, index=dates).shift(5).fillna(0.0)
        eps = 0.15 * sig + rng.normal(0, 0.006, len(dates))
        # mild autocorrelation
        for i in range(1, len(eps)):
            eps[i] += 0.08 * eps[i-1]
        apwc = 0.05 + 0.08 * pd.Series(sig, index=dates).rolling(20, min_periods=1).mean() + rng.normal(0, 0.025, len(dates))
        apwc = np.clip(apwc, -0.2, 0.8)
        for d, e, a in zip(dates, eps, apwc):
            theme_rows.append({config.DATE_COL: d, config.THEME_COL: theme, config.RESIDUAL_RETURN_COL: e})
            apwc_rows.append({config.DATE_COL: d, config.THEME_COL: theme, config.APWC_COL: a})

    theme_resid = pd.DataFrame(theme_rows)
    theme_apwc = pd.DataFrame(apwc_rows)

    return intensity[[config.DATE_COL, config.NARRATIVE_COL, "I_negative", "I_total", "I_positive"]], theme_resid, theme_apwc, None

def should_use_demo(config: Config) -> bool:
    if config.USE_DEMO_DATA is True:
        return True
    if config.USE_DEMO_DATA is False:
        return False
    # auto
    data_dir = config.DATA_DIR
    required_a = data_dir / config.NARRATIVE_INTENSITY_FILE
    constituent = data_dir / config.CONSTITUENT_RESIDUAL_FILE
    theme_resid = data_dir / config.THEME_RESIDUAL_FILE
    theme_apwc = data_dir / config.THEME_APWC_FILE
    has_theme_level = theme_resid.exists() and theme_apwc.exists()
    return not (required_a.exists() and (constituent.exists() or has_theme_level))

## 4. データ読込

実データが存在すれば読み込みます。存在しなければ synthetic demo data を使います。

実データで実行するときは、下記のようなファイル配置を想定しています。

```text
./data/daily_narrative_intensity.csv
./data/constituent_residual_returns.csv
```

または

```text
./data/daily_narrative_intensity.csv
./data/theme_residual_returns.csv
./data/theme_apwc.csv
```

In [ ]:
use_demo = should_use_demo(CONFIG)
print(f"USE_DEMO_DATA = {use_demo}")

if use_demo:
    narrative_raw, theme_resid_raw, theme_apwc_raw, constituent_raw = generate_demo_data(CONFIG)
else:
    narrative_raw = read_table(CONFIG.DATA_DIR / CONFIG.NARRATIVE_INTENSITY_FILE)
    narrative_raw = normalize_id_columns(narrative_raw, CONFIG)

    constituent_path = CONFIG.DATA_DIR / CONFIG.CONSTITUENT_RESIDUAL_FILE
    if constituent_path.exists():
        constituent_raw = read_table(constituent_path)
        constituent_raw = normalize_id_columns(constituent_raw, CONFIG)
        theme_resid_raw = None
        theme_apwc_raw = None
    else:
        constituent_raw = None
        theme_resid_raw = read_table(CONFIG.DATA_DIR / CONFIG.THEME_RESIDUAL_FILE)
        theme_apwc_raw = read_table(CONFIG.DATA_DIR / CONFIG.THEME_APWC_FILE)
        theme_resid_raw = normalize_id_columns(theme_resid_raw, CONFIG)
        theme_apwc_raw = normalize_id_columns(theme_apwc_raw, CONFIG)

print("narrative_raw:", narrative_raw.shape)
if constituent_raw is not None:
    print("constituent_raw:", constituent_raw.shape)
else:
    print("theme_resid_raw:", theme_resid_raw.shape)
    print("theme_apwc_raw:", theme_apwc_raw.shape)

display(narrative_raw.head())

## 5. Narrative intensity の整形と attention shock 作成

ここでは、long / wide どちらの形式でも受けられるように narrative intensity を正規化します。

作成する主な変数:

- `D1`: 日次 intensity の前日差分。narrative beta 推定用。
- `Shock_20D`, `Shock_60D`, `Shock_120D`: 直近平均 − その前の平均。
- `ZShock_20D`, `ZShock_60D`, `ZShock_120D`: narrative ごとの rolling z-score。
- `Smooth_20D`, `Smooth_60D`, `Smooth_120D`: narrative intensity が滑らかに上昇しているか。

In [ ]:
def normalize_narrative_intensity(df: pd.DataFrame, config: Config) -> pd.DataFrame:
    df = normalize_id_columns(df, config)
    if config.DATE_COL not in df.columns:
        raise ValueError(f"date column '{config.DATE_COL}' is required.")

    df = parse_date_col(df, config.DATE_COL)

    # Long format
    if config.NARRATIVE_COL in df.columns:
        out = df.copy()
        # intensity column fallback
        if config.INTENSITY_COL not in out.columns:
            candidates = ["I_negative", "I_total", "I_positive", "intensity", "value"]
            found = [c for c in candidates if c in out.columns]
            if not found:
                raise ValueError(f"Cannot find intensity column. Available columns: {out.columns.tolist()}")
            warnings.warn(f"{config.INTENSITY_COL} not found. Using {found[0]}.")
            out[config.INTENSITY_COL] = out[found[0]]
        return out

    # Wide format: all non-date numeric columns are narratives
    value_cols = [c for c in df.columns if c != config.DATE_COL and pd.api.types.is_numeric_dtype(df[c])]
    if not value_cols:
        raise ValueError("Wide format was inferred but no numeric narrative columns were found.")
    out = df.melt(id_vars=[config.DATE_COL], value_vars=value_cols,
                  var_name=config.NARRATIVE_COL, value_name=config.INTENSITY_COL)
    return out

def prepare_narrative_features(narrative_df: pd.DataFrame, config: Config) -> pd.DataFrame:
    df = normalize_narrative_intensity(narrative_df, config)
    df = df[[config.DATE_COL, config.NARRATIVE_COL] + [c for c in df.columns if c not in [config.DATE_COL, config.NARRATIVE_COL]]].copy()
    df[config.INTENSITY_COL] = pd.to_numeric(df[config.INTENSITY_COL], errors="coerce")
    df = df.dropna(subset=[config.DATE_COL, config.NARRATIVE_COL, config.INTENSITY_COL])
    df = df.sort_values([config.NARRATIVE_COL, config.DATE_COL])

    # narrative universe filter
    obs = df.groupby(config.NARRATIVE_COL)[config.INTENSITY_COL].agg(["count", "mean"])
    eligible = obs[obs["count"] >= config.MIN_NARRATIVE_OBS].sort_values("mean", ascending=False)
    if len(eligible) > config.MAX_NARRATIVES:
        selected = eligible.head(config.MAX_NARRATIVES).index
        df = df[df[config.NARRATIVE_COL].isin(selected)].copy()
        print(f"Narratives filtered to top {config.MAX_NARRATIVES} by average intensity.")
    else:
        print(f"Narratives used: {len(eligible)}")

    df["D1"] = df.groupby(config.NARRATIVE_COL)[config.INTENSITY_COL].diff()

    for w in config.SHOCK_WINDOWS:
        recent = df.groupby(config.NARRATIVE_COL)[config.INTENSITY_COL].transform(
            lambda x: x.shift(1).rolling(w, min_periods=max(5, w // 3)).mean()
        )
        prev = df.groupby(config.NARRATIVE_COL)[config.INTENSITY_COL].transform(
            lambda x: x.shift(1 + w).rolling(w, min_periods=max(5, w // 3)).mean()
        )
        df[f"Shock_{w}D"] = recent - prev
        df[f"ZShock_{w}D"] = df.groupby(config.NARRATIVE_COL)[f"Shock_{w}D"].transform(
            lambda x: rolling_zscore(x, config.SHOCK_Z_WINDOW, config.SHOCK_Z_MIN_PERIODS)
        )

        # frog-in-the-pan proxy
        def smoothness(x: pd.Series) -> pd.Series:
            dx = x.diff()
            mean_abs = dx.shift(1).rolling(w, min_periods=max(5, w // 3)).mean().abs()
            sd = dx.shift(1).rolling(w, min_periods=max(5, w // 3)).std()
            return mean_abs / sd.replace(0, np.nan)

        df[f"Smooth_{w}D"] = df.groupby(config.NARRATIVE_COL)[config.INTENSITY_COL].transform(smoothness)

    return df

narrative = prepare_narrative_features(narrative_raw, CONFIG)
print(narrative.shape)
display(narrative.head())

### Narrative intensity / shock の確認

平均 intensity の大きい narrative と、直近時点で shock が大きい narrative を確認します。  
実データでは、ここでニュース量が少なすぎる narrative や、異常値が多い narrative を除外してください。

In [ ]:
latest_date = narrative[CONFIG.DATE_COL].max()

summary_narr = (
    narrative.groupby(CONFIG.NARRATIVE_COL)
    .agg(
        mean_intensity=(CONFIG.INTENSITY_COL, "mean"),
        non_null=("D1", "count"),
        latest_zshock_60D=("ZShock_60D", lambda x: x.dropna().iloc[-1] if x.dropna().size else np.nan),
    )
    .sort_values("mean_intensity", ascending=False)
)
display(summary_narr.head(20))

# Plot top narratives by average intensity
top_narratives = summary_narr.head(6).index.tolist()
fig, ax = plt.subplots(figsize=(12, 5))
for n in top_narratives:
    tmp = narrative[narrative[CONFIG.NARRATIVE_COL] == n]
    ax.plot(tmp[CONFIG.DATE_COL], tmp[CONFIG.INTENSITY_COL], label=n, linewidth=1)
ax.set_title(f"Daily narrative intensity: {CONFIG.INTENSITY_COL}")
ax.set_xlabel("Date")
ax.set_ylabel("Intensity")
ax.legend()
plt.show()

## 6. テーマ残差リターンとAPWCの作成

ここでは2通りに対応します。

1. `constituent_residual_returns.csv` がある場合  
   個別銘柄残差リターンから、テーマ残差リターンとAPWCを計算します。

2. `theme_residual_returns.csv` と `theme_apwc.csv` がある場合  
   既存のテーマ別データをそのまま使います。

APWCは、テーマ内の個別銘柄残差リターンの平均ペアワイズ相関です。  
この指標は Candes et al. 型の「テーマ内の residual co-movement」を測る中核指標です。

In [ ]:
def average_pairwise_corr(window_df: pd.DataFrame, min_obs: int) -> float:
    """Average upper-triangular pairwise correlation for a window date x securities."""
    # Require at least two securities with enough observations
    valid_cols = window_df.columns[window_df.notna().sum(axis=0) >= min_obs]
    if len(valid_cols) < 2:
        return np.nan
    corr = window_df[valid_cols].corr(min_periods=min_obs)
    vals = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool)).stack()
    if vals.empty:
        return np.nan
    return vals.mean()

def compute_theme_from_constituents(constituent_df: pd.DataFrame, config: Config):
    df = normalize_id_columns(constituent_df, config)
    required = [config.DATE_COL, config.THEME_COL, config.SECURITY_COL, config.RESIDUAL_RETURN_COL]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required constituent columns: {missing}")

    df = parse_date_col(df, config.DATE_COL)
    df[config.RESIDUAL_RETURN_COL] = pd.to_numeric(df[config.RESIDUAL_RETURN_COL], errors="coerce")
    df = df.dropna(subset=[config.RESIDUAL_RETURN_COL])

    if config.WEIGHT_COL not in df.columns:
        df[config.WEIGHT_COL] = 1.0

    # Normalize weights within date-theme
    df[config.WEIGHT_COL] = pd.to_numeric(df[config.WEIGHT_COL], errors="coerce").fillna(0.0)
    weight_sum = df.groupby([config.DATE_COL, config.THEME_COL])[config.WEIGHT_COL].transform(lambda x: np.sum(np.abs(x)))
    df["_w_norm"] = np.where(weight_sum > 0, df[config.WEIGHT_COL] / weight_sum, 0.0)

    # Bottom-up residual return
    df["_weighted_resid"] = df["_w_norm"] * df[config.RESIDUAL_RETURN_COL]
    theme_resid = (
        df.groupby([config.DATE_COL, config.THEME_COL])
        .agg(
            residual_return=("_weighted_resid", "sum"),
            n_constituents=(config.SECURITY_COL, "nunique"),
        )
        .reset_index()
    )

    # APWC
    apwc_rows = []
    for theme, gdf in df.groupby(config.THEME_COL):
        pivot = (
            gdf.pivot_table(index=config.DATE_COL, columns=config.SECURITY_COL, values=config.RESIDUAL_RETURN_COL, aggfunc="mean")
            .sort_index()
        )
        dates = pivot.index
        vals = []
        for i, d in enumerate(dates):
            start = max(0, i - config.APWC_WINDOW + 1)
            win = pivot.iloc[start:i+1]
            if len(win) < config.APWC_MIN_OBS:
                vals.append(np.nan)
            else:
                vals.append(average_pairwise_corr(win, min_obs=config.APWC_MIN_OBS))
        apwc_rows.append(pd.DataFrame({
            config.DATE_COL: dates,
            config.THEME_COL: theme,
            "APWC": vals,
        }))
    theme_apwc = pd.concat(apwc_rows, ignore_index=True)

    return theme_resid, theme_apwc

def prepare_theme_features(theme_resid_df: pd.DataFrame, theme_apwc_df: pd.DataFrame, config: Config) -> pd.DataFrame:
    tr = normalize_id_columns(theme_resid_df, config)
    ta = normalize_id_columns(theme_apwc_df, config)

    tr = parse_date_col(tr, config.DATE_COL)
    ta = parse_date_col(ta, config.DATE_COL)

    tr[config.RESIDUAL_RETURN_COL] = pd.to_numeric(tr[config.RESIDUAL_RETURN_COL], errors="coerce")
    ta[config.APWC_COL] = pd.to_numeric(ta[config.APWC_COL], errors="coerce")

    df = tr.merge(
        ta[[config.DATE_COL, config.THEME_COL, config.APWC_COL]],
        on=[config.DATE_COL, config.THEME_COL],
        how="left",
    ).sort_values([config.THEME_COL, config.DATE_COL])

    # APWC z-score
    df["APWC_z"] = df.groupby(config.THEME_COL)[config.APWC_COL].transform(
        lambda x: rolling_zscore(x, config.APWC_Z_WINDOW, max(60, config.APWC_Z_WINDOW // 2))
    )

    # Residual momentum and vol
    for w in config.RMOM_WINDOWS:
        df[f"rmom_{w}D"] = df.groupby(config.THEME_COL)[config.RESIDUAL_RETURN_COL].transform(lambda x: past_sum(x, w))
    for w in config.VOL_WINDOWS:
        df[f"vol_{w}D"] = df.groupby(config.THEME_COL)[config.RESIDUAL_RETURN_COL].transform(lambda x: past_vol(x, w))

    # Future residual returns and future APWC changes
    for h in config.FUTURE_HORIZONS:
        df[f"fut_resid_ret_{h}D"] = df.groupby(config.THEME_COL)[config.RESIDUAL_RETURN_COL].transform(lambda x: future_sum(x, h))
        df[f"fut_apwc_z_{h}D"] = df.groupby(config.THEME_COL)["APWC_z"].shift(-h)
        df[f"fut_apwc_z_delta_{h}D"] = df[f"fut_apwc_z_{h}D"] - df["APWC_z"]

    return df

if constituent_raw is not None:
    theme_resid_calc, theme_apwc_calc = compute_theme_from_constituents(constituent_raw, CONFIG)
else:
    theme_resid_calc, theme_apwc_calc = theme_resid_raw, theme_apwc_raw

theme = prepare_theme_features(theme_resid_calc, theme_apwc_calc, CONFIG)
print(theme.shape)
display(theme.head())

## 7. RRG 指標の作成

RRGは、テーマの相対力と、その相対力のモメンタムを2軸で見ます。

ここでは、テーマ残差リターンから residual NAV を作り、テーマ平均の residual NAV をベンチマークとして、以下を計算します。

\[
RS_{g,t}=\frac{NAV^{resid}_{g,t}}{NAV^{resid}_{bench,t}}
\]

\[
RSRatio_{g,t}=Z(\log RS_{g,t})
\]

\[
RSMomentum_{g,t}=Z(\log RS_{g,t}-\log RS_{g,t-L})
\]

状態分類は以下です。

| State | 条件 |
|---|---|
| Leading | RS-Ratio > 0 and RS-Momentum > 0 |
| Weakening | RS-Ratio > 0 and RS-Momentum <= 0 |
| Lagging | RS-Ratio <= 0 and RS-Momentum <= 0 |
| Improving | RS-Ratio <= 0 and RS-Momentum > 0 |

In [ ]:
def add_rrg_features(theme_df: pd.DataFrame, config: Config) -> pd.DataFrame:
    df = theme_df.sort_values([config.THEME_COL, config.DATE_COL]).copy()

    # residual NAV by theme
    df["_log_ret"] = safe_log1p(df[config.RESIDUAL_RETURN_COL])
    df["resid_nav"] = df.groupby(config.THEME_COL)["_log_ret"].cumsum().pipe(np.exp)

    # equal-weight benchmark residual return across themes
    bench = (
        df.groupby(config.DATE_COL)[config.RESIDUAL_RETURN_COL]
        .mean()
        .rename("bench_resid_return")
        .reset_index()
        .sort_values(config.DATE_COL)
    )
    bench["bench_nav"] = np.exp(safe_log1p(bench["bench_resid_return"]).cumsum())

    df = df.merge(bench[[config.DATE_COL, "bench_nav"]], on=config.DATE_COL, how="left")
    df["log_rs"] = np.log(df["resid_nav"] / df["bench_nav"])

    df["rs_ratio"] = df.groupby(config.THEME_COL)["log_rs"].transform(
        lambda x: rolling_zscore(x, config.RRG_WINDOW, max(30, config.RRG_WINDOW // 2))
    )
    df["_rs_mom_raw"] = df.groupby(config.THEME_COL)["log_rs"].transform(lambda x: x - x.shift(config.RRG_MOM_LAG))
    df["rs_momentum"] = df.groupby(config.THEME_COL)["_rs_mom_raw"].transform(
        lambda x: rolling_zscore(x, config.RRG_WINDOW, max(30, config.RRG_WINDOW // 2))
    )

    conditions = [
        (df["rs_ratio"] > 0) & (df["rs_momentum"] > 0),
        (df["rs_ratio"] > 0) & (df["rs_momentum"] <= 0),
        (df["rs_ratio"] <= 0) & (df["rs_momentum"] <= 0),
        (df["rs_ratio"] <= 0) & (df["rs_momentum"] > 0),
    ]
    states = ["Leading", "Weakening", "Lagging", "Improving"]
    df["RRG_state"] = np.select(conditions, states, default="Unknown")

    for h in config.FUTURE_HORIZONS:
        df[f"RRG_state_fut_{h}D"] = df.groupby(config.THEME_COL)["RRG_state"].shift(-h)
        df[f"to_leading_{h}D"] = (df[f"RRG_state_fut_{h}D"] == "Leading").astype(float)

    return df.drop(columns=["_log_ret", "_rs_mom_raw"], errors="ignore")

theme = add_rrg_features(theme, CONFIG)
display(theme[[CONFIG.DATE_COL, CONFIG.THEME_COL, CONFIG.RESIDUAL_RETURN_COL, "APWC_z", "rmom_60D", "rs_ratio", "rs_momentum", "RRG_state"]].head(10))

## 8. テーマ別 Narrative beta と Narrative Signal の作成

ここでは、daily intensity の変化に対するテーマ残差リターンの rolling beta を推定します。

基本形:

\[
\epsilon_{g,t}=\alpha_{g,n}+\gamma_{g,n}\Delta I_{n,t}+u_{g,n,t}
\]

推定した \(\gamma_{g,n,t}\) と narrative shock を掛け合わせて、テーマ別 Narrative Signal を作ります。

\[
NarrativeSignal^{J}_{g,t}=\sum_n\gamma_{g,n,t}ZShock^{J}_{n,t}
\]

この変数が、後続の APWC や residual return を予測するかを検証します。

In [ ]:
def rolling_beta_pair(y: pd.Series, x: pd.Series, window: int, min_periods: int) -> pd.Series:
    """Rolling OLS beta for y on x, shifted by 1 day to avoid look-ahead."""
    aligned = pd.concat([y, x], axis=1).dropna()
    if aligned.empty:
        return pd.Series(index=y.index, dtype=float)

    yy = aligned.iloc[:, 0]
    xx = aligned.iloc[:, 1]

    mean_x = xx.rolling(window, min_periods=min_periods).mean()
    mean_y = yy.rolling(window, min_periods=min_periods).mean()
    mean_xy = (xx * yy).rolling(window, min_periods=min_periods).mean()
    mean_x2 = (xx * xx).rolling(window, min_periods=min_periods).mean()

    cov_xy = mean_xy - mean_x * mean_y
    var_x = mean_x2 - mean_x * mean_x
    beta = cov_xy / var_x.replace(0, np.nan)

    # beta at t uses data through t-1
    beta = beta.shift(1)
    return beta.reindex(y.index)

def build_narrative_signals(theme_df: pd.DataFrame, narrative_df: pd.DataFrame, config: Config):
    # Pivot theme residual returns
    y_wide = (
        theme_df.pivot_table(index=config.DATE_COL, columns=config.THEME_COL, values=config.RESIDUAL_RETURN_COL, aggfunc="mean")
        .sort_index()
    )

    # Narrative driver for beta
    driver = (
        narrative_df.pivot_table(index=config.DATE_COL, columns=config.NARRATIVE_COL, values=config.BETA_DRIVER_COL, aggfunc="mean")
        .sort_index()
    )

    common_dates = y_wide.index.intersection(driver.index)
    y_wide = y_wide.loc[common_dates]
    driver = driver.loc[common_dates]

    # Shock matrices
    shock_mats = {}
    for w in config.SHOCK_WINDOWS:
        col = f"ZShock_{w}D"
        shock_mats[w] = (
            narrative_df.pivot_table(index=config.DATE_COL, columns=config.NARRATIVE_COL, values=col, aggfunc="mean")
            .reindex(common_dates)
            .sort_index()
        )

    signal_by_w = {w: pd.DataFrame(0.0, index=common_dates, columns=y_wide.columns) for w in config.SHOCK_WINDOWS}
    latest_betas = []

    themes = y_wide.columns.tolist()
    narratives = driver.columns.tolist()

    for theme_id in themes:
        y = y_wide[theme_id]
        for narrative_id in narratives:
            x = driver[narrative_id]
            beta = rolling_beta_pair(y, x, config.BETA_WINDOW, config.BETA_MIN_PERIODS)

            # Keep latest beta for heatmap / diagnostics
            if beta.dropna().size:
                latest_betas.append({
                    config.THEME_COL: theme_id,
                    config.NARRATIVE_COL: narrative_id,
                    "latest_beta": beta.dropna().iloc[-1],
                    "mean_abs_beta": beta.abs().mean(),
                })

            for w, shock in shock_mats.items():
                if narrative_id in shock.columns:
                    signal_by_w[w][theme_id] = signal_by_w[w][theme_id] + beta.fillna(0.0) * shock[narrative_id].fillna(0.0)

    # Long format signals
    signal_long = None
    for w, mat in signal_by_w.items():
        tmp = (
            mat.stack()
            .rename(f"narr_signal_{w}D")
            .reset_index()
            .rename(columns={"level_0": config.DATE_COL, "level_1": config.THEME_COL})
        )
        if signal_long is None:
            signal_long = tmp
        else:
            signal_long = signal_long.merge(tmp, on=[config.DATE_COL, config.THEME_COL], how="outer")

    beta_diag = pd.DataFrame(latest_betas)

    # Cross-sectional ranks for signal robustness
    for w in config.SHOCK_WINDOWS:
        col = f"narr_signal_{w}D"
        rank_col = f"{col}_xrank"
        signal_long[rank_col] = signal_long.groupby(config.DATE_COL)[col].transform(
            lambda x: x.rank(pct=True) - 0.5
        )

    return signal_long, beta_diag

narr_signals, beta_diag = build_narrative_signals(theme, narrative, CONFIG)
print(narr_signals.shape)
display(narr_signals.head())
display(beta_diag.sort_values("mean_abs_beta", ascending=False).head(20))

### テーマ×Narrative beta のヒートマップ

ここでは最新時点の narrative beta を確認します。  
実データでは、テーマと自然に対応する narrative が上位に出ているかを必ず目視してください。

例:

- AIテーマ → AI / Semiconductors / Cloud  
- Defenseテーマ → Defense / International Conflicts  
- Banksテーマ → Interest Rates / Liquidity  
- Nuclearテーマ → Nuclear / Energy Security

In [ ]:
if not beta_diag.empty:
    top_pairs = beta_diag.copy()
    # Heatmap用に平均絶対betaが大きいnarrativeを選ぶ
    top_narr_for_heatmap = (
        top_pairs.groupby(CONFIG.NARRATIVE_COL)["mean_abs_beta"]
        .mean()
        .sort_values(ascending=False)
        .head(15)
        .index
    )
    heat = (
        top_pairs[top_pairs[CONFIG.NARRATIVE_COL].isin(top_narr_for_heatmap)]
        .pivot(index=CONFIG.THEME_COL, columns=CONFIG.NARRATIVE_COL, values="latest_beta")
        .fillna(0.0)
    )

    fig, ax = plt.subplots(figsize=(14, max(4, 0.4 * len(heat))))
    im = ax.imshow(heat.values, aspect="auto")
    ax.set_xticks(np.arange(len(heat.columns)))
    ax.set_xticklabels(heat.columns, rotation=45, ha="right")
    ax.set_yticks(np.arange(len(heat.index)))
    ax.set_yticklabels(heat.index)
    ax.set_title("Latest theme × narrative beta")
    fig.colorbar(im, ax=ax, fraction=0.02)
    plt.tight_layout()
    plt.show()
else:
    print("No beta diagnostics available.")

## 9. 分析用パネルの作成

テーマ特徴量と Narrative Signal を結合し、回帰・ソート・イベントスタディ用のパネルを作ります。

主な列:

- `narr_signal_20D`, `narr_signal_60D`, `narr_signal_120D`
- `APWC_z`
- `rmom_20D`, `rmom_60D`, `rmom_120D`
- `fut_resid_ret_5D`, `fut_resid_ret_20D`, `fut_resid_ret_60D`
- `fut_apwc_z_delta_5D`, `fut_apwc_z_delta_20D`, `fut_apwc_z_delta_60D`
- `RRG_state`, `to_leading_20D`

In [ ]:
panel = theme.merge(narr_signals, on=[CONFIG.DATE_COL, CONFIG.THEME_COL], how="left")
panel["month"] = panel[CONFIG.DATE_COL].dt.to_period("M").astype(str)

# Interaction variables
for w in CONFIG.SHOCK_WINDOWS:
    sig = f"narr_signal_{w}D"
    if sig in panel.columns:
        panel[f"{sig}_x_APWC_z"] = panel[sig] * panel["APWC_z"]
        panel[f"{sig}_x_RMOM60"] = panel[sig] * panel.get("rmom_60D", np.nan)

print(panel.shape)
display(panel.head())
display(panel[[CONFIG.DATE_COL, CONFIG.THEME_COL, CONFIG.MAIN_SIGNAL_COL, "APWC_z", "rmom_60D", "fut_resid_ret_20D", "fut_apwc_z_delta_20D"]].dropna().head())

## 10. Panel regression の関数

日次パネルでは、テーマごとの固定効果や日付効果を考慮する必要があります。  
ここでは、簡易的に **テーマ固定効果 + 日付固定効果** を two-way demeaning で除去し、OLSを行います。

標準誤差は、利用可能なら `statsmodels` の cluster robust covariance を使います。  
日次データで horizon が重複するため、厳密には Newey-West / Driscoll-Kraay / two-way cluster も検討してください。

In [ ]:
def iterative_two_way_demean(df: pd.DataFrame, cols: list[str], fe1: str, fe2: str, n_iter: int = 10) -> pd.DataFrame:
    x = df[cols].astype(float).copy()
    grand = x.mean()
    for _ in range(n_iter):
        x = x - x.groupby(df[fe1]).transform("mean")
        x = x - x.groupby(df[fe2]).transform("mean")
        x = x + grand
    return x

def fit_panel_ols(
    df: pd.DataFrame,
    y_col: str,
    x_cols: list[str],
    fe_theme: bool = True,
    fe_date: bool = True,
    date_fe_col: str = "month",
    cluster_col: Optional[str] = None,
):
    needed = [y_col] + x_cols + [CONFIG.THEME_COL, date_fe_col]
    data = df[needed].replace([np.inf, -np.inf], np.nan).dropna().copy()

    if data.empty or len(data) < 50:
        raise ValueError(f"Not enough observations for {y_col} ~ {x_cols}")

    cols = [y_col] + x_cols

    if fe_theme and fe_date:
        dm = iterative_two_way_demean(data, cols, CONFIG.THEME_COL, date_fe_col)
        y = dm[y_col]
        X = dm[x_cols]
        add_const = False
    elif fe_theme:
        dm = data[cols] - data.groupby(CONFIG.THEME_COL)[cols].transform("mean")
        y = dm[y_col]
        X = dm[x_cols]
        add_const = False
    elif fe_date:
        dm = data[cols] - data.groupby(date_fe_col)[cols].transform("mean")
        y = dm[y_col]
        X = dm[x_cols]
        add_const = False
    else:
        y = data[y_col]
        X = data[x_cols]
        add_const = True

    X = X.astype(float)
    y = y.astype(float)

    if HAS_STATSMODELS:
        if add_const:
            X_fit = sm.add_constant(X)
        else:
            X_fit = X

        model = sm.OLS(y, X_fit, missing="drop")
        if cluster_col is not None and cluster_col in data.columns:
            res = model.fit(cov_type="cluster", cov_kwds={"groups": data[cluster_col]})
        else:
            res = model.fit(cov_type="HAC", cov_kwds={"maxlags": 5})
        return res, data
    else:
        # Fallback simple OLS
        X_np = X.values
        y_np = y.values
        beta = np.linalg.lstsq(X_np, y_np, rcond=None)[0]
        return pd.Series(beta, index=x_cols), data

def regression_table(results: dict) -> pd.DataFrame:
    rows = []
    for name, res in results.items():
        if HAS_STATSMODELS:
            for param in res.params.index:
                rows.append({
                    "model": name,
                    "variable": param,
                    "coef": res.params[param],
                    "t": res.tvalues[param],
                    "p": res.pvalues[param],
                    "nobs": res.nobs,
                    "r2": res.rsquared,
                })
        else:
            for param, coef in res.items():
                rows.append({"model": name, "variable": param, "coef": coef})
    return pd.DataFrame(rows)

## 11. 回帰分析 1：Narrative Signal が将来APWCを予測するか

仮説:

\[
NarrativeSignal_{g,t} \Rightarrow APWCZ_{g,t+h}
\]

または、

\[
NarrativeSignal_{g,t} \Rightarrow \Delta APWCZ_{g,t:t+h}
\]

ここで有意な正の係数が出れば、ニュース上の narrative shock がテーマ内残差相関の上昇に先行している可能性があります。

In [ ]:
results_apwc = {}

for h in CONFIG.FUTURE_HORIZONS:
    y_col = f"fut_apwc_z_delta_{h}D"
    if y_col not in panel.columns:
        continue

    x_cols = [
        CONFIG.MAIN_SIGNAL_COL,
        "APWC_z",
        CONFIG.MAIN_RMOM_COL,
    ]
    x_cols = [x for x in x_cols if x in panel.columns]

    try:
        res, used = fit_panel_ols(panel, y_col, x_cols, fe_theme=True, fe_date=True, date_fe_col="month")
        results_apwc[f"APWC_delta_{h}D"] = res
    except Exception as e:
        print(f"Skipped {h}D: {e}")

apwc_reg_table = regression_table(results_apwc)
display(apwc_reg_table)

## 12. 回帰分析 2：Narrative Signal が将来のテーマ残差リターンを予測するか

仮説:

\[
NarrativeSignal_{g,t} \Rightarrow R^{resid}_{g,t+1:t+h}
\]

これは、テーマバスケット版 Narrative Momentum の中核検証です。

In [ ]:
results_ret = {}

for h in CONFIG.FUTURE_HORIZONS:
    y_col = f"fut_resid_ret_{h}D"
    if y_col not in panel.columns:
        continue

    x_cols = [
        CONFIG.MAIN_SIGNAL_COL,
        "APWC_z",
        CONFIG.MAIN_RMOM_COL,
        "vol_60D" if "vol_60D" in panel.columns else None,
    ]
    x_cols = [x for x in x_cols if x is not None and x in panel.columns]

    try:
        res, used = fit_panel_ols(panel, y_col, x_cols, fe_theme=True, fe_date=True, date_fe_col="month")
        results_ret[f"FutureResidualReturn_{h}D"] = res
    except Exception as e:
        print(f"Skipped {h}D: {e}")

ret_reg_table = regression_table(results_ret)
display(ret_reg_table)

## 13. 回帰分析 3：Narrative Signal × APWC の相互作用

最も重要な統合仮説は次です。

\[
R^{resid}_{g,t+1:t+h}
=
\alpha_g+\lambda_t
+\beta_1 NarrativeSignal_{g,t}
+\beta_2 APWCZ_{g,t}
+\beta_3 RMOM_{g,t}
+\beta_4 NarrativeSignal_{g,t}\times APWCZ_{g,t}
+u_{g,t+h}
\]

\[
\beta_4>0
\]

なら、ニュース上の narrative shock と価格上の残差共振が同時にあるテーマほど、将来の residual return momentum が強いことを示唆します。

In [ ]:
results_interaction = {}

for h in CONFIG.FUTURE_HORIZONS:
    y_col = f"fut_resid_ret_{h}D"
    inter_col = f"{CONFIG.MAIN_SIGNAL_COL}_x_APWC_z"

    x_cols = [
        CONFIG.MAIN_SIGNAL_COL,
        "APWC_z",
        CONFIG.MAIN_RMOM_COL,
        inter_col,
    ]
    x_cols = [x for x in x_cols if x in panel.columns]

    try:
        res, used = fit_panel_ols(panel, y_col, x_cols, fe_theme=True, fe_date=True, date_fe_col="month")
        results_interaction[f"Interaction_{h}D"] = res
    except Exception as e:
        print(f"Skipped {h}D: {e}")

interaction_reg_table = regression_table(results_interaction)
display(interaction_reg_table)

## 14. RRG 遷移分析

ここでは、Narrative Signal がテーマの RRG 状態遷移、特に **Leading 入り** を予測するかを見ます。

簡易には線形確率モデルで、

\[
1(RRG_{g,t+h}=Leading)
=
\alpha_g+\lambda_t
+\beta NarrativeSignal_{g,t}
+\delta APWCZ_{g,t}
+\eta RMOM_{g,t}
+u_{g,t+h}
\]

を推定します。

本格的には logit / multinomial logit / Markov transition model も候補です。

In [ ]:
results_rrg = {}

for h in CONFIG.FUTURE_HORIZONS:
    y_col = f"to_leading_{h}D"
    if y_col not in panel.columns:
        continue

    x_cols = [
        CONFIG.MAIN_SIGNAL_COL,
        "APWC_z",
        CONFIG.MAIN_RMOM_COL,
        "rs_ratio",
        "rs_momentum",
    ]
    x_cols = [x for x in x_cols if x in panel.columns]

    try:
        res, used = fit_panel_ols(panel, y_col, x_cols, fe_theme=True, fe_date=True, date_fe_col="month")
        results_rrg[f"ToLeading_{h}D_LPM"] = res
    except Exception as e:
        print(f"Skipped {h}D: {e}")

rrg_reg_table = regression_table(results_rrg)
display(rrg_reg_table)

## 15. クロスセクション・ソート

回帰だけでなく、各日付でテーマを Narrative Signal によって並べ、将来リターンや将来APWC変化を比較します。

主に確認するもの:

1. Narrative Signal 上位テーマは将来 residual return が高いか  
2. Narrative Signal 上位テーマは将来 APWC が上がるか  
3. APWCが高い群の中で Narrative Signal の効果が強いか

In [ ]:
def cross_sectional_sort_test(
    df: pd.DataFrame,
    signal_col: str,
    outcome_col: str,
    n_quantiles: int = 5,
    date_col: str = CONFIG.DATE_COL,
) -> pd.DataFrame:
    data = df[[date_col, CONFIG.THEME_COL, signal_col, outcome_col]].replace([np.inf, -np.inf], np.nan).dropna().copy()

    def assign_q(x):
        try:
            return pd.qcut(x, q=n_quantiles, labels=False, duplicates="drop") + 1
        except Exception:
            return pd.Series(np.nan, index=x.index)

    data["q"] = data.groupby(date_col)[signal_col].transform(assign_q)
    data = data.dropna(subset=["q"])
    data["q"] = data["q"].astype(int)

    qret = (
        data.groupby(["q", date_col])[outcome_col]
        .mean()
        .reset_index()
        .groupby("q")[outcome_col]
        .agg(["mean", "std", "count"])
        .reset_index()
    )
    qret["t"] = qret["mean"] / (qret["std"] / np.sqrt(qret["count"]))

    # spread top - bottom by date
    pivot = data.groupby([date_col, "q"])[outcome_col].mean().unstack("q")
    if 1 in pivot.columns and n_quantiles in pivot.columns:
        spread = pivot[n_quantiles] - pivot[1]
        spread_stats = pd.DataFrame({
            "q": [f"{n_quantiles}-1"],
            "mean": [spread.mean()],
            "std": [spread.std()],
            "count": [spread.count()],
            "t": [spread.mean() / (spread.std() / np.sqrt(spread.count())) if spread.count() > 1 else np.nan],
        })
        qret = pd.concat([qret, spread_stats], ignore_index=True)

    return qret

sort_ret = cross_sectional_sort_test(panel, CONFIG.MAIN_SIGNAL_COL, CONFIG.MAIN_FUTURE_RETURN_COL, n_quantiles=5)
sort_apwc = cross_sectional_sort_test(panel, CONFIG.MAIN_SIGNAL_COL, CONFIG.MAIN_FUTURE_APWC_DELTA_COL, n_quantiles=5)

print("Sort: future residual return")
display(sort_ret)
print("Sort: future APWC delta")
display(sort_apwc)

## 16. APWC High / Low 別の Narrative Signal ソート

ここでは、APWCが高い局面で Narrative Signal の予測力が強まるかを見ます。

方法:

1. 各日付でテーマを APWC_z により high / low に分割  
2. それぞれの群内で Narrative Signal によりソート  
3. 将来 residual return の上位−下位スプレッドを比較  

この結果は、`Narrative Signal × APWC` の相互作用を直感的に説明する表になります。

In [ ]:
def conditional_sort_by_apwc(
    df: pd.DataFrame,
    signal_col: str,
    outcome_col: str,
    apwc_col: str = "APWC_z",
    n_quantiles: int = 3,
    date_col: str = CONFIG.DATE_COL,
) -> pd.DataFrame:
    data = df[[date_col, CONFIG.THEME_COL, signal_col, outcome_col, apwc_col]].replace([np.inf, -np.inf], np.nan).dropna().copy()

    # high/low APWC by date median
    data["apwc_group"] = data.groupby(date_col)[apwc_col].transform(lambda x: np.where(x >= x.median(), "High_APWC", "Low_APWC"))

    def assign_q(x):
        try:
            return pd.qcut(x, q=n_quantiles, labels=False, duplicates="drop") + 1
        except Exception:
            return pd.Series(np.nan, index=x.index)

    data["q"] = data.groupby([date_col, "apwc_group"])[signal_col].transform(assign_q)
    data = data.dropna(subset=["q"])
    data["q"] = data["q"].astype(int)

    rows = []
    for group, gdf in data.groupby("apwc_group"):
        pivot = gdf.groupby([date_col, "q"])[outcome_col].mean().unstack("q")
        if 1 in pivot.columns and n_quantiles in pivot.columns:
            spread = pivot[n_quantiles] - pivot[1]
            rows.append({
                "apwc_group": group,
                "spread": f"Q{n_quantiles}-Q1",
                "mean": spread.mean(),
                "std": spread.std(),
                "count": spread.count(),
                "t": spread.mean() / (spread.std() / np.sqrt(spread.count())) if spread.count() > 1 else np.nan,
            })
    return pd.DataFrame(rows)

cond_sort = conditional_sort_by_apwc(panel, CONFIG.MAIN_SIGNAL_COL, CONFIG.MAIN_FUTURE_RETURN_COL, n_quantiles=3)
display(cond_sort)

## 17. イベントスタディ

Narrative Signal が非常に高いテーマ日をイベントとし、その前後で APWC_z と累積残差リターンがどう動くかを確認します。

期待される順序は以下です。

\[
NarrativeShock \rightarrow APWC上昇 \rightarrow Residual Momentum形成 \rightarrow RRG Improving/Leading
\]

In [ ]:
def event_study(
    df: pd.DataFrame,
    event_col: str,
    value_cols: list[str],
    threshold_quantile: float = 0.9,
    pre: int = 20,
    post: int = 60,
    max_events_per_theme: Optional[int] = None,
):
    data = df.sort_values([CONFIG.THEME_COL, CONFIG.DATE_COL]).copy()
    threshold = data[event_col].quantile(threshold_quantile)
    events = data[data[event_col] >= threshold][[CONFIG.DATE_COL, CONFIG.THEME_COL, event_col]].dropna()

    # Optional: reduce event clustering by taking top events per theme
    if max_events_per_theme is not None:
        events = (
            events.sort_values([CONFIG.THEME_COL, event_col], ascending=[True, False])
            .groupby(CONFIG.THEME_COL)
            .head(max_events_per_theme)
        )

    rows = []
    for _, ev in events.iterrows():
        theme_id = ev[CONFIG.THEME_COL]
        event_date = ev[CONFIG.DATE_COL]
        gdf = data[data[CONFIG.THEME_COL] == theme_id].reset_index(drop=True)
        idx_arr = np.where(gdf[CONFIG.DATE_COL].values == np.datetime64(event_date))[0]
        if len(idx_arr) == 0:
            continue
        idx = idx_arr[0]
        start = max(0, idx - pre)
        end = min(len(gdf) - 1, idx + post)
        win = gdf.iloc[start:end+1].copy()
        win["tau"] = np.arange(start - idx, end - idx + 1)
        win["event_date"] = event_date
        win["event_theme"] = theme_id
        rows.append(win[["event_date", "event_theme", "tau"] + value_cols])

    if not rows:
        return pd.DataFrame()

    long = pd.concat(rows, ignore_index=True)
    avg = long.groupby("tau")[value_cols].mean().reset_index()
    count = long.groupby("tau").size().rename("n_events").reset_index()
    avg = avg.merge(count, on="tau", how="left")
    return avg

# cumulative future/residual path around event: create rolling cumulative residual around event
panel_for_event = panel.copy()
panel_for_event["cum_resid_nav_log"] = panel_for_event.groupby(CONFIG.THEME_COL)[CONFIG.RESIDUAL_RETURN_COL].cumsum()

event_avg = event_study(
    panel_for_event,
    event_col=CONFIG.MAIN_SIGNAL_COL,
    value_cols=["APWC_z", "cum_resid_nav_log", CONFIG.MAIN_SIGNAL_COL],
    threshold_quantile=0.9,
    pre=20,
    post=60,
    max_events_per_theme=20,
)

display(event_avg.head())

if not event_avg.empty:
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(event_avg["tau"], event_avg["APWC_z"], label="Average APWC_z")
    ax.axvline(0, linestyle="--")
    ax.set_title("Event study: APWC_z around high Narrative Signal events")
    ax.set_xlabel("Event time")
    ax.legend()
    plt.show()

    fig, ax = plt.subplots(figsize=(10, 5))
    # normalize cumulative log NAV to tau=0
    base = event_avg.loc[event_avg["tau"] == 0, "cum_resid_nav_log"]
    base_val = base.iloc[0] if len(base) else 0
    ax.plot(event_avg["tau"], event_avg["cum_resid_nav_log"] - base_val, label="Cumulative residual return path")
    ax.axvline(0, linestyle="--")
    ax.set_title("Event study: cumulative residual return around high Narrative Signal events")
    ax.set_xlabel("Event time")
    ax.legend()
    plt.show()

## 18. RRG 可視化

RRG 平面上にテーマを配置し、Narrative Signal の強さで色を付けます。

- 右上: Leading  
- 左上: Improving  
- 左下: Lagging  
- 右下: Weakening  

Narrative Signal が高いテーマが Improving / Leading に集中しているかを見ると、ニュースと相対力ローテーションの関係を直感的に確認できます。

In [ ]:
plot_date = panel[CONFIG.DATE_COL].dropna().max()
rrg_snapshot = panel[panel[CONFIG.DATE_COL] == plot_date].dropna(subset=["rs_ratio", "rs_momentum", CONFIG.MAIN_SIGNAL_COL]).copy()

if not rrg_snapshot.empty:
    fig, ax = plt.subplots(figsize=(9, 7))
    sc = ax.scatter(rrg_snapshot["rs_ratio"], rrg_snapshot["rs_momentum"], c=rrg_snapshot[CONFIG.MAIN_SIGNAL_COL], s=80)
    for _, row in rrg_snapshot.iterrows():
        ax.text(row["rs_ratio"], row["rs_momentum"], str(row[CONFIG.THEME_COL]), fontsize=8, alpha=0.8)
    ax.axhline(0, linestyle="--")
    ax.axvline(0, linestyle="--")
    ax.set_xlabel("RS-Ratio")
    ax.set_ylabel("RS-Momentum")
    ax.set_title(f"RRG snapshot colored by Narrative Signal: {plot_date.date()}")
    fig.colorbar(sc, ax=ax, label=CONFIG.MAIN_SIGNAL_COL)
    plt.show()
else:
    print("No RRG snapshot available.")

## 19. 統合ポートフォリオ・ソートの簡易バックテスト

ここでは、月次リバランスで以下のルールを試します。

### Long-only coherence-gated narrative strategy

各月末に、以下を満たすテーマを候補にします。

1. `NarrativeSignal > 0`
2. `APWC_z > 0`
3. `Residual Momentum > 0`

候補の中で Narrative Signal 上位を等ウェイトします。  
候補がない場合はキャッシュです。

これは、以下の仮説をそのまま投資ルールにしたものです。

**ニュース上の盛り上がり + 残差共振 + 残差モメンタムが同時にあるテーマを保有する。**

In [ ]:
def max_drawdown(nav: pd.Series) -> float:
    peak = nav.cummax()
    dd = nav / peak - 1.0
    return dd.min()

def performance_stats(ret: pd.Series, periods_per_year: int = 252) -> pd.Series:
    ret = ret.dropna()
    if ret.empty:
        return pd.Series(dtype=float)
    mean = ret.mean() * periods_per_year
    vol = ret.std() * np.sqrt(periods_per_year)
    sr = mean / vol if vol > 0 else np.nan
    nav = (1 + ret).cumprod()
    return pd.Series({
        "ann_mean": mean,
        "ann_vol": vol,
        "sharpe": sr,
        "max_drawdown": max_drawdown(nav),
        "n_days": ret.count(),
    })

def monthly_coherence_gated_strategy(
    df: pd.DataFrame,
    signal_col: str,
    rmom_col: str,
    max_themes: int = 5,
):
    data = df.sort_values([CONFIG.DATE_COL, CONFIG.THEME_COL]).copy()
    dates = sorted(data[CONFIG.DATE_COL].unique())

    # month-end rebalance dates available in panel
    date_frame = pd.DataFrame({CONFIG.DATE_COL: pd.to_datetime(dates)})
    date_frame["month"] = date_frame[CONFIG.DATE_COL].dt.to_period("M")
    rebalance_dates = date_frame.groupby("month")[CONFIG.DATE_COL].max().tolist()

    weights = []
    for d in rebalance_dates:
        snap = data[data[CONFIG.DATE_COL] == d].dropna(subset=[signal_col, "APWC_z", rmom_col])
        candidates = snap[(snap[signal_col] > 0) & (snap["APWC_z"] > 0) & (snap[rmom_col] > 0)]
        candidates = candidates.sort_values(signal_col, ascending=False).head(max_themes)
        if len(candidates) == 0:
            continue
        w = 1.0 / len(candidates)
        for theme_id in candidates[CONFIG.THEME_COL]:
            weights.append({CONFIG.DATE_COL: d, CONFIG.THEME_COL: theme_id, "weight": w})

    weights = pd.DataFrame(weights)
    if weights.empty:
        return pd.DataFrame(), pd.Series(dtype=float)

    # apply weights from next business day until next rebalance
    ret_wide = data.pivot_table(index=CONFIG.DATE_COL, columns=CONFIG.THEME_COL, values=CONFIG.RESIDUAL_RETURN_COL, aggfunc="mean").sort_index()
    weight_wide = weights.pivot_table(index=CONFIG.DATE_COL, columns=CONFIG.THEME_COL, values="weight", aggfunc="sum").reindex(ret_wide.index)
    weight_wide = weight_wide.ffill().fillna(0.0)

    # use previous day's weights for today's returns
    strategy_ret = (weight_wide.shift(1).reindex(columns=ret_wide.columns).fillna(0.0) * ret_wide).sum(axis=1)
    return weights, strategy_ret

weights, strat_ret = monthly_coherence_gated_strategy(panel, CONFIG.MAIN_SIGNAL_COL, CONFIG.MAIN_RMOM_COL, max_themes=5)
stats = performance_stats(strat_ret)
display(stats)

if not strat_ret.empty:
    nav = (1 + strat_ret.fillna(0)).cumprod()
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(nav.index, nav.values)
    ax.set_title("Monthly coherence-gated narrative strategy: residual-return NAV")
    ax.set_xlabel("Date")
    ax.set_ylabel("NAV")
    plt.show()

display(weights.tail(20))

## 20. 結果の保存

回帰表、パネル、Narrative Signal、ソート結果などを `outputs` フォルダに保存します。

In [ ]:
CONFIG.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Parquet は pyarrow / fastparquet が必要なため、依存関係を増やさないよう gzip CSV で保存します。
# 大規模データで Parquet を使いたい場合は、pyarrow をインストールして to_parquet に変更してください。
narrative.to_csv(CONFIG.OUTPUT_DIR / "prepared_narrative_features.csv.gz", index=False, compression="gzip")
theme.to_csv(CONFIG.OUTPUT_DIR / "prepared_theme_features.csv.gz", index=False, compression="gzip")
narr_signals.to_csv(CONFIG.OUTPUT_DIR / "theme_narrative_signals.csv.gz", index=False, compression="gzip")
panel.to_csv(CONFIG.OUTPUT_DIR / "analysis_panel.csv.gz", index=False, compression="gzip")

apwc_reg_table.to_csv(CONFIG.OUTPUT_DIR / "regression_apwc.csv", index=False)
ret_reg_table.to_csv(CONFIG.OUTPUT_DIR / "regression_future_residual_return.csv", index=False)
interaction_reg_table.to_csv(CONFIG.OUTPUT_DIR / "regression_interaction.csv", index=False)
rrg_reg_table.to_csv(CONFIG.OUTPUT_DIR / "regression_rrg_transition.csv", index=False)

sort_ret.to_csv(CONFIG.OUTPUT_DIR / "sort_future_residual_return.csv", index=False)
sort_apwc.to_csv(CONFIG.OUTPUT_DIR / "sort_future_apwc_delta.csv", index=False)
cond_sort.to_csv(CONFIG.OUTPUT_DIR / "conditional_sort_apwc.csv", index=False)

if not beta_diag.empty:
    beta_diag.to_csv(CONFIG.OUTPUT_DIR / "theme_narrative_beta_diagnostics.csv", index=False)

print(f"Saved outputs to: {CONFIG.OUTPUT_DIR.resolve()}")

## 21. 実証結果を読む際のチェックリスト

実データで分析した後は、以下を確認してください。

### Narrative intensity 側

- 特定 narrative の intensity が構造的にゼロに近すぎないか
- ニュースソースや記事数の増減で intensity が歪んでいないか
- negative / total / positive intensity のどれが効いているか
- 20D / 60D / 120D shock のどれが最も安定しているか

### Theme beta 側

- テーマと自然に対応する narrative beta が出ているか
- beta が極端に不安定なテーマ・narrative がないか
- rolling window を変えても符号や強度が安定するか

### APWC 先行性

- `NarrativeSignal -> future APWC delta` が有意か
- 現在の APWC_z をコントロールしても効くか
- APWC high / low で効果が異なるか

### Residual Momentum 先行性

- `NarrativeSignal -> future residual return` が有意か
- `RMOM` を入れても Narrative Signal が残るか
- `NarrativeSignal × APWC_z` が正で有意か

### RRG

- Narrative Signal が Improving → Leading を予測するか
- Leading に入った後のリターンが高いか
- Weakening への移行で利食いシグナルになるか

### 実運用化

- シグナルは必ずリターン観測より前に固定する
- 月末3営業日前などの point-in-time ルールを決める
- テーマウェイト、構成銘柄、Barra exposure はすべて過去時点のものを使う
- 取引コスト、売買回転率、テーマ重複を別途評価する